In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import XLNetTokenizer
import sys
import os

# Add src to path (damit er deine Skripte findet)
sys.path.append(os.path.abspath('..'))

from src.dataset import get_datasets
from src.model import SingleHeadXLNet

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 1. Daten laden (Wir nutzen nur einen Mini-Teil)
tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')

# Beachte: get_datasets gibt jetzt 4 Werte zurück!
# Wir nutzen mode='clarity' für den Test
train_ds, _, _, _ = get_datasets(
    '../data/raw/train.csv', 
    '../data/raw/test.csv', 
    tokenizer,
    mode='clarity'
)

# Wir nehmen nur die ersten 8 Beispiele
mini_batch = torch.utils.data.Subset(train_ds, range(8))
loader = DataLoader(mini_batch, batch_size=4, shuffle=True)

# Batch checken
batch = next(iter(loader))
print("Input shape:", batch['input_ids'].shape)
print("Labels:", batch['labels']) # Sollten 0, 1 oder 2 sein

# 2. Modell initialisieren (Single Head, 3 Klassen)
model = SingleHeadXLNet(num_labels=3).to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # Hohe LR zum schnellen Overfit
criterion = nn.CrossEntropyLoss()

print("\n--- Starting Sanity Check (Overfitting) ---")

for epoch in range(30):
    total_loss = 0
    for batch in loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass (Single Head)
        logits = model(ids, mask)
        
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    if epoch % 5 == 0:
        print(f"Epoch {epoch}: Loss {total_loss:.4f}")

print(f"Final Loss: {total_loss:.4f}")
# Wenn das < 0.1 oder nahe 0 ist, funktioniert dein Code!

Device: cpu
--- Loading Datasets (Mode: clarity) ---
Applying Democratic Voting Logic to Test Set...
Train size: 3448 | Test size: 308
Input shape: torch.Size([4, 512])
Labels: tensor([0, 0, 1, 1])
loading xlnet-base-cased with 3 labels...

--- Starting Sanity Check (Overfitting) ---
Epoch 0: Loss 6.7966
Epoch 5: Loss 1.4259
Epoch 10: Loss 1.3046
Epoch 15: Loss 1.2254
Epoch 20: Loss 1.0305
Epoch 25: Loss 0.8823
Final Loss: 0.8521
